In [6]:
# 사용할 라이브러리
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

In [7]:
# 데이터 불러오기

# 공정 데이터
secom = pd.read_csv("secom.csv", sep=r"\s+", header=None)   
# 공정 데이터 라벨
secom_labels = pd.read_csv("secom_labels.csv", sep=r"\s+", header=None) 

In [ ]:
# 컬럼명 지정

# 0행 1열
# 공정 데이터 컬럼명 지정. shape[1] == 열의 갯수 
secom.columns = [f"feature+{i}" for i in range(secom.shape[1])] 
# 공정 데이터 라벨 컬럼명 지정
secom_labels.columns = ["label", "datetime"]    

# 데이터를 '열'방향으로 이어 붙힘 
df = pd.concat([secom, secom_labels], axis=1) 

# "label" 데이터 값에 대한 갯수 -1 pass, 1 fail
print(df["label"].value_counts()) 
# 불량인 데이터 출력
print(df[df["label"] == 1]) 
# 데이터 합격 / 불량 비율
print(df["label"].value_counts(normalize=True) * 100) 

In [9]:
# 결측치 확인

# 'feature+' 로 시작하는 컬럼 묶음
feature_cols = [col for col in df.columns if str(col).startswith("feature+")] 

# isnull() -> null이면 1, 데이터가 있으면 0
# sum() -> 합계 
# 둘이 합치면 마치 count()처럼 작동.

total_missing_data = df[feature_cols].isnull().sum().sum()
print("결측치 개수 : ",total_missing_data)

결측치 개수 :  41951


In [14]:
# 결측치가 있는 feature 개수
missing_feature_count = (df[feature_cols].isnull().sum() > 0).sum()
print("결측치가 있는 feature 개수:", missing_feature_count)

결측치가 있는 feature 개수: 538


In [ ]:
# 결측치가 많은 feature들을 내림차순으로 출력
top20_missing_feature = df[feature_cols].isnull().sum().sort_values(ascending=False).head(20) 
print("결측치가 많은 feature :\n",top20_missing_feature)

In [ ]:
# 1. 통과 / 실패 비율

# 통과들만 모아놓은 dataframe
pass_df = df[df["label"] == -1]

# 실패들만 모아놓은 dataframe
fail_df = df[df["label"] == 1]

# 통과값들의 평균값
pass_mean = pass_df[feature_cols].mean()
# 실패값들의 평균값
fail_mean = fail_df[feature_cols].mean()
# 전체값들의 표준편차
std_values = df[feature_cols].std()

# 차이가 큰 값들부터 출력
mean_diff_score = ((fail_mean - pass_mean).abs() / std_values).sort_values(ascending=False) * 100
print(mean_diff_score)

In [17]:
# 2. 값이 거의 변하지 않는 feature 확인
nunique_feature = df[feature_cols].nunique(axis=1).head(20)
print(nunique_feature)

0     405
1     404
2     427
3     428
4     406
5     424
6     412
7     419
8     425
9     418
10    424
11    408
12    420
13    420
14    415
15    415
16    425
17    413
18    420
19    420
dtype: int64


In [ ]:
# 해야 할 일.
# 1. 통과 / 실패 비율
# 2. 값이 거의 변하지 않는 feature 확인 -> 정상 / 불량을 확인하는데 정보가 거의 없기 때문임.
# 3. 정상과 불량에서 차이가 큰 feature 

In [ ]:
# 결측치 처리
# 모델 성능을 위한 결측값을 '중간값'으로 대체

# 중간값으로 하는 이유 : 평균값은 극단값에 흔들릴 수 있음.

# 인스턴스 생성
imputer = SimpleImputer(missing_values= np.nan, strategy='median')

# 결측치가 있는 데이터는 'feature+~'이므로 feature_cols 사용, label/datetime 제외
X = df[feature_cols]

# transform 메서드를 이용하여 결측치를 '중간값'으로 대체
transform_X_data = imputer.fit_transform(X)

transform_X_data = pd.DataFrame(transform_X_data, columns=feature_cols)
print("결측치 갯수 : ", transform_X_data.isnull().sum().sum())
print(transform_X_data)